# GALVAT Trial 9 — Clean-Data Training, Fair Re-Benchmark, and Detector + Tracker Packaging

Trial 9 is intentionally **not** just Trial 8 with more epochs.

The main changes are:

- train from the fully QA-cleaned Roboflow export;
- preserve original image resolution in the export when possible and let YOLO letterbox internally;
- merge complementary labels across byte-identical duplicate images instead of simply dropping one copy;
- build a brand-new multilabel-stratified train/validation/test split after cleanup;
- freeze the new test set before model selection;
- select checkpoint, NMS, and per-class confidence thresholds using validation GALVAT only;
- evaluate Trial 8 and Trial 9 on the **same new frozen test set** for an apples-to-apples 44-class GALVAT comparison;
- report any new class such as `measuring spoon` separately until it has an official GALVAT rating;
- package YOLO26s as the semantic detector and package a separate SAM 3 tracking contract rather than fusing the two models.

## Deployment architecture

**YOLO26s remains the semantic authority.** It decides *what* the object is.

**SAM 3 is temporal localization/tracking only.** A YOLO box seeds a SAM 3 track; SAM 3 propagates location/mask between detector refreshes. SAM 3 does not overwrite the YOLO class label.

The intended runtime loop is:

`camera → YOLO26s → per-class thresholds → choose ≤3 useful detections → seed/refresh SAM 3 tracks → concise assistive output`

YOLO is rerun periodically or when a track becomes uncertain. This prevents SAM drift from silently changing the semantic identity of an object.

The export bundle contains a detector-only deployment profile plus a CUDA SAM-3 reference integration profile. The latter is kept separate because the official SAM 3 video predictor is CUDA/PyTorch-oriented, while Magic Leap 2 is an x86_64 Android device with an AMD Navi 2 GPU.

In [ ]:
# ============================================================
# 1. ENVIRONMENT + CONFIG
# ============================================================

!pip -q install roboflow ultralytics==8.4.102 onnx onnxruntime pyyaml pandas tqdm pillow iterative-stratification

from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
import datetime
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import yaml
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive, userdata
drive.mount("/content/drive")

import torch
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Select a GPU runtime first."

GPU_NAME = torch.cuda.get_device_name(0)

print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("GPU:", GPU_NAME)
print(
    "GPU memory GB:",
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
)

# ============================================================
# DOWNLOAD ROBOFLOW VERSION 2 DIRECTLY
# ============================================================

rf = Roboflow(
    api_key=userdata.get("ROBOFLOW_API_KEY")
)

project = (
    rf.workspace("aud-research")
    .project("objects-ow83t-ylj7g")
)

version = project.version(2)

dataset = version.download(
    "yolo26",
    location="/content/galvat_trial9_rf_v2",
    overwrite=True
)

ROBOFLOW_ROOT = Path(dataset.location)

print("Roboflow dataset downloaded to:")
print(ROBOFLOW_ROOT)

# ============================================================
# TRIAL 9 CONFIG
# ============================================================

TRIAL8_PT = Path(
    "/content/drive/MyDrive/galvat_trial8_training_runs/"
    "galvat_trial8_yolo26s_640_galvat_selected_2026-08-03_v1/"
    "export/galvat_trial8_selected.pt"
)

WORK_ROOT = Path("/content/galvat_trial9_split")

DRIVE_RUNS = Path(
    "/content/drive/MyDrive/galvat_trial9_training_runs"
)

RUN_NAME = "galvat_trial9_yolo26s_rf_v2"
RUN_ROOT = DRIVE_RUNS / RUN_NAME

AUDIT_DIR = RUN_ROOT / "dataset_audit"
TRAIN_DIR = RUN_ROOT / "train"
SELECTION_DIR = RUN_ROOT / "validation_selection"
TEST_DIR = RUN_ROOT / "frozen_test"
EXPORT_DIR = RUN_ROOT / "export"
BUNDLE_DIR = RUN_ROOT / "deployment_bundle"

for p in [
    RUN_ROOT,
    AUDIT_DIR,
    SELECTION_DIR,
    TEST_DIR,
    EXPORT_DIR,
    BUNDLE_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

MODEL_INIT = "yolo26s.pt"

IMGSZ = 640
EPOCHS = 160
SAVE_PERIOD = 10

TRAIN_FRACTION = 0.70
VALID_FRACTION = 0.15
TEST_FRACTION = 0.15

MATCH_IOU = 0.50
NMS_IOU_VALUES = [0.45, 0.55, 0.65]

CONF_VALUES = [
    round(float(x), 3)
    for x in np.arange(0.025, 0.901, 0.025)
]

PRED_MIN_CONF = 0.01
MAX_DET = 300
PRED_BATCH = 8

print("Dataset root:", ROBOFLOW_ROOT)
print("Run root:", RUN_ROOT)

## Dataset expectation

For Trial 9, the cleanest export is an original-resolution YOLO dataset generated from the corrected Roboflow source annotations.

If the final export is still pre-baked to 640×640 with black bars, the notebook can train on it, but it will warn you. Preserving native source resolution is preferable because it avoids permanently baking padding into the training imagery and retains more information for small assistive objects.

In [ ]:
# ============================================================
# 2. DISCOVER ROBOFLOW DATASET + YAML
# ============================================================

yaml_candidates = (
    list(ROBOFLOW_ROOT.rglob("data.yaml"))
    + list(ROBOFLOW_ROOT.rglob("data.yml"))
)

assert yaml_candidates, (
    f"No data.yaml found under {ROBOFLOW_ROOT}"
)

SOURCE_YAML = sorted(
    yaml_candidates,
    key=lambda p: (len(p.parts), str(p))
)[0]

SOURCE_ROOT = SOURCE_YAML.parent

with SOURCE_YAML.open() as f:
    cfg = yaml.safe_load(f)

names_obj = cfg["names"]

if isinstance(names_obj, dict):
    keys = sorted(
        names_obj,
        key=lambda x: int(x)
    )

    NAMES = [
        str(names_obj[k])
        for k in keys
    ]

else:
    NAMES = [
        str(x)
        for x in names_obj
    ]


def norm_name(name):
    x = str(name).strip().lower()
    x = x.replace("/", " ")
    x = x.replace("-", " ")
    x = x.replace("_", " ")
    x = re.sub(r"\s+", " ", x)
    return x


def metric_name(name):
    return norm_name(name).replace(" ", "_")


print("Source root:", SOURCE_ROOT)
print("Source YAML:", SOURCE_YAML)
print("Classes:", len(NAMES))

for i, name in enumerate(NAMES):
    print(i, name)

if "measuring spoon" in {
    norm_name(x)
    for x in NAMES
}:
    print("✅ measuring spoon is present.")
else:
    print("⚠️ measuring spoon was not found.")

In [ ]:
# ============================================================
# 3. DISCOVER ALL SOURCE IMAGES/LABELS
# ============================================================

def label_path_for_image(image_path):
    parts = list(Path(image_path).parts)

    for i, part in enumerate(parts):
        if part == "images":
            parts[i] = "labels"
            return Path(*parts).with_suffix(".txt")

    raise ValueError(
        f"Cannot infer label path from {image_path}"
    )


def all_source_images():
    out = []

    for split_name in [
        "train", "valid", "val", "test"
    ]:
        image_dir = SOURCE_ROOT / split_name / "images"

        if image_dir.exists():
            out.extend(
                p
                for p in image_dir.iterdir()
                if p.is_file()
                and p.suffix.lower() in IMAGE_EXTS
            )

    # Some exports use paths that are not the standard split layout.
    if not out:
        out = [
            p
            for p in SOURCE_ROOT.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTS
            and "images" in p.parts
        ]

    return sorted(set(out))


def parse_yolo_file(label_path):
    label_path = Path(label_path)

    if not label_path.exists():
        return []

    text = label_path.read_text().strip()

    if not text:
        return []

    rows = []

    for line in text.splitlines():
        parts = line.split()

        if len(parts) < 5:
            raise ValueError(
                f"Bad YOLO row in {label_path}: {line}"
            )

        cid = int(float(parts[0]))
        values = list(map(float, parts[1:]))

        # ----------------------------------------------------
        # Normal YOLO detection box:
        # class xc yc w h
        # ----------------------------------------------------
        if len(values) == 4:
            x, y, w, h = values

        # ----------------------------------------------------
        # Polygon / segmentation annotation:
        # class x1 y1 x2 y2 x3 y3 ...
        #
        # Convert polygon to its enclosing bounding box.
        # ----------------------------------------------------
        elif len(values) >= 6 and len(values) % 2 == 0:
            xs = values[0::2]
            ys = values[1::2]

            x1 = min(xs)
            y1 = min(ys)
            x2 = max(xs)
            y2 = max(ys)

            x = (x1 + x2) / 2.0
            y = (y1 + y2) / 2.0
            w = x2 - x1
            h = y2 - y1

        else:
            raise ValueError(
                f"Unrecognized YOLO row in {label_path}: {line}"
            )

        # Safety clamp
        x = min(max(x, 0.0), 1.0)
        y = min(max(y, 0.0), 1.0)
        w = min(max(w, 0.0), 1.0)
        h = min(max(h, 0.0), 1.0)

        if w <= 0 or h <= 0:
            continue

        rows.append(
            (cid, x, y, w, h)
        )

    return rows


def sha1_file(path, chunk=1024 * 1024):
    h = hashlib.sha1()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


source_images = all_source_images()

print("Source images:", f"{len(source_images):,}")

assert source_images, "No images found."

## Exact duplicates are handled differently than Trial 8

Trial 8 removed duplicate images by split precedence.

Trial 9 instead **unions complementary annotations across byte-identical images**, then keeps one canonical copy. This matters for GALVAT because the dataset originated from many single-class sources: two identical images can legitimately carry different useful labels.

Same-class boxes that substantially overlap are deduplicated; different-class boxes are preserved.

In [ ]:
# ============================================================
# 4. EXACT-IMAGE DEDUPE + COMPLEMENTARY LABEL UNION
# ============================================================

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

UNIQUE_ROOT = WORK_ROOT / "unique"
UNIQUE_IMAGES = UNIQUE_ROOT / "images"
UNIQUE_LABELS = UNIQUE_ROOT / "labels"

UNIQUE_IMAGES.mkdir(parents=True)
UNIQUE_LABELS.mkdir(parents=True)


def xywh_to_xyxy(row):
    cid, x, y, w, h = row

    return np.array([
        x - w / 2,
        y - h / 2,
        x + w / 2,
        y + h / 2,
    ], dtype=float)


def box_iou(a, b):
    tl = np.maximum(a[:2], b[:2])
    br = np.minimum(a[2:], b[2:])
    wh = np.maximum(0.0, br - tl)
    inter = wh[0] * wh[1]

    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def containment_over_smaller(a, b):
    tl = np.maximum(a[:2], b[:2])
    br = np.minimum(a[2:], b[2:])
    wh = np.maximum(0.0, br - tl)
    inter = wh[0] * wh[1]

    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])

    smaller = min(area_a, area_b)

    return inter / smaller if smaller > 0 else 0.0


def union_label_rows(label_lists):
    merged = []

    for rows in label_lists:
        for row in rows:
            cid = row[0]
            box = xywh_to_xyxy(row)

            same_object = False

            for old in merged:
                if old[0] != cid:
                    continue

                old_box = xywh_to_xyxy(old)

                if (
                    box_iou(box, old_box) >= 0.80
                    or containment_over_smaller(box, old_box) >= 0.92
                ):
                    same_object = True
                    break

            if not same_object:
                merged.append(row)

    return merged


hash_groups = defaultdict(list)

for image_path in tqdm(
    source_images,
    desc="Hashing source images"
):
    hash_groups[
        sha1_file(image_path)
    ].append(image_path)


dedupe_manifest = []
unique_records = []

for image_hash, paths in tqdm(
    hash_groups.items(),
    total=len(hash_groups),
    desc="Merging exact duplicate labels"
):
    label_lists = [
        parse_yolo_file(
            label_path_for_image(p)
        )
        for p in paths
    ]

    merged_rows = union_label_rows(
        label_lists
    )

    canonical = paths[0]
    ext = canonical.suffix.lower()

    out_stem = (
        image_hash[:20]
        + "__"
        + re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            canonical.stem
        )[:60]
    )

    out_image = UNIQUE_IMAGES / (out_stem + ext)
    out_label = UNIQUE_LABELS / (out_stem + ".txt")

    shutil.copy2(
        canonical,
        out_image
    )

    lines = [
        f"{cid} {x:.8f} {y:.8f} {w:.8f} {h:.8f}"
        for cid, x, y, w, h in merged_rows
    ]

    out_label.write_text(
        "\n".join(lines)
        + ("\n" if lines else "")
    )

    unique_records.append({
        "image_hash": image_hash,
        "image_path": str(out_image),
        "label_path": str(out_label),
        "box_count": len(merged_rows),
        "class_ids": sorted({
            row[0]
            for row in merged_rows
        }),
    })

    dedupe_manifest.append({
        "image_hash": image_hash,
        "copies_found": len(paths),
        "source_paths": " | ".join(
            str(p)
            for p in paths
        ),
        "source_box_counts": " | ".join(
            str(len(x))
            for x in label_lists
        ),
        "merged_box_count": len(merged_rows),
    })


unique_df = pd.DataFrame(
    unique_records
)

dedupe_df = pd.DataFrame(
    dedupe_manifest
)

dedupe_df.to_csv(
    AUDIT_DIR / "exact_dedupe_label_union.csv",
    index=False
)

print("Source images:", f"{len(source_images):,}")
print("Unique byte-identical images:", f"{len(unique_df):,}")
print(
    "Exact duplicate copies removed:",
    f"{len(source_images) - len(unique_df):,}"
)
print(
    "Duplicate hash groups:",
    f"{int((dedupe_df['copies_found'] > 1).sum()):,}"
)

In [ ]:
# ============================================================
# 5. IMAGE-SIZE AUDIT
# ============================================================

size_rows = []

for row in tqdm(
    unique_df.itertuples(index=False),
    total=len(unique_df),
    desc="Auditing source image sizes"
):
    with Image.open(row.image_path) as im:
        w, h = im.size

    size_rows.append({
        "image_path": row.image_path,
        "width": w,
        "height": h,
        "is_640_square": (
            w == 640 and h == 640
        ),
    })


size_df = pd.DataFrame(size_rows)

size_df.to_csv(
    AUDIT_DIR / "source_image_sizes.csv",
    index=False
)

square_rate = float(
    size_df["is_640_square"].mean()
)

print(
    "640x640 image fraction:",
    f"{square_rate:.2%}"
)

if square_rate > 0.90:
    print(
        "WARNING: nearly the entire export is already 640x640. "
        "If these images contain baked black Fit padding, consider "
        "regenerating the final Roboflow export without fixed resize."
    )
else:
    print(
        "✅ Source resolutions are varied; YOLO can perform its own "
        "640px letterboxing during training."
    )

## New split

The cleaned dataset is re-split from scratch using multilabel stratification based on **class presence per image**.

The split is created once and then frozen by manifest hash. The test set is never used for checkpoint, NMS, or confidence-threshold selection.

In [ ]:
# ============================================================
# 6. MULTILABEL-STRATIFIED 70/15/15 SPLIT
# ============================================================

Y = np.zeros(
    (len(unique_df), len(NAMES)),
    dtype=int
)

for i, class_ids in enumerate(
    unique_df["class_ids"]
):
    for cid in class_ids:
        Y[i, int(cid)] = 1


all_idx = np.arange(
    len(unique_df)
)

first_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=(
        VALID_FRACTION
        + TEST_FRACTION
    ),
    random_state=SEED,
)

train_idx, hold_idx = next(
    first_splitter.split(
        all_idx,
        Y
    )
)

relative_test_fraction = (
    TEST_FRACTION
    / (
        VALID_FRACTION
        + TEST_FRACTION
    )
)

second_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=relative_test_fraction,
    random_state=SEED + 1,
)

hold_local = np.arange(
    len(hold_idx)
)

valid_local, test_local = next(
    second_splitter.split(
        hold_local,
        Y[hold_idx]
    )
)

valid_idx = hold_idx[
    valid_local
]

test_idx = hold_idx[
    test_local
]


split_assignments = {}

for i in train_idx:
    split_assignments[int(i)] = "train"

for i in valid_idx:
    split_assignments[int(i)] = "valid"

for i in test_idx:
    split_assignments[int(i)] = "test"


SPLIT_ROOT = WORK_ROOT / "dataset"

for split in [
    "train",
    "valid",
    "test"
]:
    (SPLIT_ROOT / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )
    (SPLIT_ROOT / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


split_rows = []

for i, row in tqdm(
    list(unique_df.iterrows()),
    desc="Materializing frozen split"
):
    split = split_assignments[int(i)]

    image_src = Path(
        row["image_path"]
    )

    label_src = Path(
        row["label_path"]
    )

    image_dst = (
        SPLIT_ROOT
        / split
        / "images"
        / image_src.name
    )

    label_dst = (
        SPLIT_ROOT
        / split
        / "labels"
        / label_src.name
    )

    shutil.copy2(
        image_src,
        image_dst
    )

    shutil.copy2(
        label_src,
        label_dst
    )

    split_rows.append({
        "image_hash": row["image_hash"],
        "split": split,
        "image": image_dst.name,
        "box_count": row["box_count"],
        "class_ids": ",".join(
            str(x)
            for x in row["class_ids"]
        ),
    })


split_manifest = pd.DataFrame(
    split_rows
)

SPLIT_MANIFEST = (
    AUDIT_DIR
    / "frozen_split_manifest.csv"
)

split_manifest.to_csv(
    SPLIT_MANIFEST,
    index=False
)


DATA_YAML = SPLIT_ROOT / "data.yaml"

data_yaml = {
    "path": str(SPLIT_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        i: name
        for i, name in enumerate(NAMES)
    },
    "nc": len(NAMES),
}

DATA_YAML.write_text(
    yaml.safe_dump(
        data_yaml,
        sort_keys=False
    )
)


def file_sha1(path):
    h = hashlib.sha1()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


SPLIT_MANIFEST_SHA1 = file_sha1(
    SPLIT_MANIFEST
)

print(
    split_manifest["split"]
    .value_counts()
)

print(
    "Frozen split manifest SHA1:",
    SPLIT_MANIFEST_SHA1
)

In [ ]:
# ============================================================
# 7. SPLIT / CLASS COVERAGE AUDIT
# ============================================================

def split_images(split):
    return sorted(
        p
        for p in (
            SPLIT_ROOT
            / split
            / "images"
        ).iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
    )


class_rows = []

for cid, class_name in enumerate(NAMES):
    row = {
        "class_id": cid,
        "class": class_name,
    }

    for split in [
        "train",
        "valid",
        "test"
    ]:
        image_count = 0
        box_count = 0

        for image_path in split_images(split):
            rows = parse_yolo_file(
                label_path_for_image(
                    image_path
                )
            )

            class_boxes = [
                x
                for x in rows
                if x[0] == cid
            ]

            if class_boxes:
                image_count += 1
                box_count += len(
                    class_boxes
                )

        row[
            f"{split}_images"
        ] = image_count

        row[
            f"{split}_boxes"
        ] = box_count

    class_rows.append(row)


class_summary = pd.DataFrame(
    class_rows
)

class_summary.to_csv(
    AUDIT_DIR
    / "frozen_split_class_support.csv",
    index=False
)

display(
    class_summary
    .sort_values(
        "train_images"
    )
)

missing_coverage = class_summary[
    (
        class_summary[
            [
                "train_images",
                "valid_images",
                "test_images"
            ]
        ] == 0
    ).any(axis=1)
]

assert missing_coverage.empty, (
    "At least one class is missing from a split. "
    "Inspect missing_coverage before training."
)

print("✅ Every class appears in train, validation, and frozen test.")

## GALVAT scoring

Trial 9 can train on the expanded ontology, but the primary comparison score remains the **44 rated Trial-8 classes** because `measuring spoon` does not yet have an official GALVAT FN/FP rating in the Trial-8 metric definition.

The notebook therefore reports:

- `shared_44_galvat`: primary apples-to-apples Trial 8 vs Trial 9 score;
- normal precision/recall/AP for every Trial-9 class, including measuring spoon.

If a formal measuring-spoon rating is later defined, it can be added to the ratings dictionary and the extended score recomputed.

In [ ]:
# ============================================================
# 8. EXACT GALVAT METRIC FROM TRIAL 8
# ============================================================

GALVAT_X = np.array(
    [0.00, 0.05, 0.10, 0.20, 0.50],
    dtype=float
)

GALVAT_RATINGS = {
    "bathroom_sign": {"fn":[8,8,8,8,9], "fp":[7,8,8,8,9]},
    "canned_good": {"fn":[2,2,2,3,4], "fp":[2,2,2,3,5]},
    "cellphone": {"fn":[10,10,10,10,10], "fp":[9,9,9,9,9]},
    "chair": {"fn":[8,8,8,9,9], "fp":[10,10,10,10,10]},
    "coffee_mug": {"fn":[8,9,9,10,10], "fp":[5,5,5,5,6]},
    "deodorant": {"fn":[6,6,7,7,8], "fp":[6,6,7,7,7]},
    "door": {"fn":[9,9,9,9,9], "fp":[10,10,10,10,10]},
    "drinking_glass": {"fn":[8,8,9,9,10], "fp":[5,5,5,5,6]},
    "electrical_outlet": {"fn":[7,7,8,8,9], "fp":[7,7,8,8,9]},
    "exit_sign": {"fn":[7,7,7,8,8], "fp":[7,8,8,8,9]},
    "eyeglasses": {"fn":[8,8,8,9,10], "fp":[3,3,4,5,6]},
    "fork": {"fn":[7,7,7,8,9], "fp":[6,7,7,8,8]},
    "hearing_aid": {"fn":[9,9,9,10,10], "fp":[4,4,5,6,7]},
    "jug": {"fn":[6,6,6,7,8], "fp":[5,5,6,6,7]},
    "knife": {"fn":[10,10,10,10,10], "fp":[6,7,7,8,8]},
    "lemon": {"fn":[4,4,5,6,7], "fp":[4,4,5,6,7]},
    "light_switch": {"fn":[7,7,8,8,9], "fp":[5,5,5,5,6]},
    "measuring_cup": {"fn":[4,4,5,6,7], "fp":[3,3,3,3,4]},
    "measuring_spoon": {"fn":[4,4,5,6,7], "fp":[3,3,3,3,4]},
    "paper_towel_dispenser": {"fn":[5,5,6,7,7], "fp":[5,5,6,7,7]},
    "pot": {"fn":[7,7,8,8,9], "fp":[5,5,5,5,6]},
    "razor": {"fn":[8,8,8,9,9], "fp":[6,7,7,8,8]},
    "salt_pepper_shaker": {"fn":[6,6,6,7,8], "fp":[2,2,2,3,4]},
    "soap_dispenser": {"fn":[6,6,7,7,8], "fp":[7,7,7,8,8]},
    "spoon": {"fn":[5,5,5,5,6], "fp":[5,5,5,5,6]},
    "stairs": {"fn":[10,10,10,10,10], "fp":[8,8,8,9,10]},
    "table": {"fn":[7,7,7,8,8], "fp":[7,7,7,8,8]},
    "thermostat": {"fn":[2,2,2,2,3], "fp":[2,2,2,2,3]},
    "toilet": {"fn":[9,9,9,10,10], "fp":[10,10,10,10,10]},
    "toothbrush": {"fn":[6,6,7,7,8], "fp":[7,7,8,8,9]},
    "toothpaste": {"fn":[8,8,9,9,9], "fp":[8,8,9,9,9]},
    "trash_can": {"fn":[5,5,5,5,6], "fp":[6,6,7,7,7]},
    "tupperware": {"fn":[3,3,3,4,5], "fp":[2,3,3,4,5]},
    "urinal": {"fn":[9,9,9,10,10], "fp":[10,10,10,10,10]},
    "wallet": {"fn":[9,9,9,9,9], "fp":[10,10,10,10,10]},
    "window": {"fn":[4,4,4,5,5], "fp":[7,8,8,8,8]},
    "wine_glass": {"fn":[8,8,9,9,10], "fp":[5,5,5,5,6]},
    "writing_device": {"fn":[2,2,3,4,4], "fp":[1,2,3,3,4]},
    "keys": {"fn":[10,10,10,10,10], "fp":[8,8,8,8,8]},
    "medicine_bottle": {"fn":[10,10,10,10,10], "fp":[10,10,10,10,10]},
    "cane": {"fn":[10,10,10,10,10], "fp":[8,8,8,8,8]},
    "tv_remote": {"fn":[4,4,5,6,7], "fp":[3,3,3,4,5]},
    "purse": {"fn":[9,9,9,9,9], "fp":[10,10,10,10,10]},
    "railing": {"fn":[8,8,8,9,9], "fp":[7,7,8,8,9]},
    "pan": {"fn":[7,7,8,8,9], "fp":[5,5,5,5,6]},
}


def auc(x, y):
    fn = getattr(
        np,
        "trapezoid",
        np.trapz
    )

    return float(
        fn(
            np.asarray(y, dtype=float),
            np.asarray(x, dtype=float)
        )
    )


def remaining_area(error_rate, ratings):
    possible = auc(
        GALVAT_X,
        ratings
    )

    error = min(
        max(float(error_rate), 0.0),
        1.0
    )

    if error >= 0.50:
        return 0.0, possible, 0.0

    xs = np.asarray([
        error,
        *[
            t
            for t in GALVAT_X
            if t > error
        ]
    ])

    ys = np.interp(
        xs,
        GALVAT_X,
        np.asarray(
            ratings,
            dtype=float
        )
    )

    remaining = auc(
        xs,
        ys
    )

    return (
        remaining,
        possible,
        remaining / possible
        if possible
        else 0.0
    )


dataset_metric_names = {
    metric_name(x)
    for x in NAMES
}

RATED_CLASSES = sorted(
    dataset_metric_names.intersection(
        GALVAT_RATINGS
    )
)

unrated_dataset_classes = sorted(
    dataset_metric_names
    - set(GALVAT_RATINGS)
)

print("Rated GALVAT classes in Trial 9:", len(RATED_CLASSES))
print("Unrated Trial-9 classes:", unrated_dataset_classes)

assert len(RATED_CLASSES) == 45, (
    "Expected all 45 Trial-9 GALVAT-rated classes, including measuring spoon."
)

assert not unrated_dataset_classes, (
    f"These dataset classes are missing GALVAT ratings: "
    f"{unrated_dataset_classes}"
)

In [ ]:
# ============================================================
# 9. TRAIN YOLO26s — TRIAL-9 RECIPE
# ============================================================

LAST_PT = TRAIN_DIR / "weights" / "last.pt"
BEST_PT = TRAIN_DIR / "weights" / "best.pt"
TRAIN_COMPLETE = TRAIN_DIR / "TRAINING_COMPLETE.json"

train_args = dict(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,

    # A100 / YOLO26s
    batch=64,
    device=0,
    workers=8,
    cache="disk",

    # Keep the optimizer family that already worked well in Trial 8,
    # while giving the cleaned dataset more time to converge.
    optimizer="AdamW",
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=4.0,
    patience=40,
    cos_lr=True,

    pretrained=True,
    freeze=0,

    # Realistic assistive-camera robustness:
    # mild framing/scale variation, light rotation, modest occlusion.
    mosaic=0.35,
    close_mosaic=20,
    mixup=0.0,
    cutmix=0.05,
    multi_scale=0.15,

    fliplr=0.0,
    flipud=0.0,

    hsv_h=0.01,
    hsv_s=0.30,
    hsv_v=0.20,

    degrees=3.0,
    translate=0.08,
    scale=0.30,
    shear=1.0,
    perspective=0.0005,

    save=True,
    save_period=SAVE_PERIOD,
    plots=True,
    amp=True,

    seed=SEED,
    deterministic=True,

    project=str(RUN_ROOT),
    name="train",
    exist_ok=True,
    verbose=True,
)

training_signature = {
    "model_init": MODEL_INIT,
    "split_manifest_sha1": SPLIT_MANIFEST_SHA1,
    "train_args": {
        k: str(v)
        for k, v in train_args.items()
    },
}

TRAIN_CONFIG_SHA1 = hashlib.sha1(
    json.dumps(
        training_signature,
        sort_keys=True
    ).encode()
).hexdigest()

training_signature[
    "configuration_sha1"
] = TRAIN_CONFIG_SHA1

TRAIN_CONFIG_PATH = (
    RUN_ROOT
    / "train_configuration.json"
)

if TRAIN_CONFIG_PATH.exists():
    prior = json.loads(
        TRAIN_CONFIG_PATH.read_text()
    )

    assert prior[
        "configuration_sha1"
    ] == TRAIN_CONFIG_SHA1, (
        "Run folder already contains a different training configuration. "
        "Increment RUN_NAME."
    )
else:
    TRAIN_CONFIG_PATH.write_text(
        json.dumps(
            training_signature,
            indent=2
        )
    )


if TRAIN_COMPLETE.exists():
    print("Training already complete.")

elif LAST_PT.exists():
    print(
        "Resuming interrupted training from:",
        LAST_PT
    )

    model = YOLO(
        str(LAST_PT)
    )

    model.train(
        resume=True
    )

    TRAIN_COMPLETE.write_text(
        json.dumps({
            "resumed": True,
            "configuration_sha1": TRAIN_CONFIG_SHA1,
            "completed_at": str(
                datetime.datetime.now()
            ),
        }, indent=2)
    )

else:
    print("Starting Trial 9 from:", MODEL_INIT)

    model = YOLO(
        MODEL_INIT
    )

    model.train(
        **train_args
    )

    TRAIN_COMPLETE.write_text(
        json.dumps({
            "resumed": False,
            "configuration_sha1": TRAIN_CONFIG_SHA1,
            "completed_at": str(
                datetime.datetime.now()
            ),
        }, indent=2)
    )


assert BEST_PT.exists()
assert LAST_PT.exists()

print("BEST_PT:", BEST_PT)
print("LAST_PT:", LAST_PT)

In [ ]:
# ============================================================
# 10. GALVAT EVALUATION HELPERS — CLASS-NAME BASED
#     (supports Trial 8 = 44 classes and Trial 9 = 45 classes)
# ============================================================

def load_ground_truth(split):
    cache = {}

    for image_path in split_images(split):
        rows = parse_yolo_file(
            label_path_for_image(
                image_path
            )
        )

        by_class = defaultdict(list)

        for row in rows:
            class_key = metric_name(
                NAMES[
                    row[0]
                ]
            )

            by_class[
                class_key
            ].append(
                xywh_to_xyxy(
                    row
                )
            )

        cache[
            image_path.name
        ] = {
            class_key: np.asarray(
                boxes,
                dtype=float
            )
            for class_key, boxes in by_class.items()
        }

    return cache


GT_CACHE = {
    split: load_ground_truth(split)
    for split in [
        "valid",
        "test"
    ]
}


def collect_predictions(
    model_path,
    split,
    nms_iou,
):
    paths = split_images(
        split
    )

    predictions = {}

    detector = YOLO(
        str(model_path)
    )

    model_names = detector.names

    progress = tqdm(
        total=len(paths),
        desc=(
            f"{Path(model_path).name} "
            f"{split} nms={nms_iou}"
        )
    )

    try:
        for start in range(
            0,
            len(paths),
            PRED_BATCH
        ):
            chunk = paths[
                start:start + PRED_BATCH
            ]

            results = detector.predict(
                source=[
                    str(x)
                    for x in chunk
                ],
                stream=False,
                imgsz=IMGSZ,
                conf=PRED_MIN_CONF,
                iou=float(nms_iou),
                max_det=MAX_DET,
                device=0,
                batch=len(chunk),
                rect=False,
                agnostic_nms=False,
                verbose=False,
            )

            assert len(results) == len(chunk)

            for source_path, result in zip(
                chunk,
                results
            ):
                key = Path(
                    source_path
                ).name

                rows = []

                if (
                    result.boxes is not None
                    and len(result.boxes)
                ):
                    class_ids = (
                        result.boxes.cls
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(int)
                    )

                    confidences = (
                        result.boxes.conf
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(float)
                    )

                    boxes = (
                        result.boxes.xyxyn
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(float)
                    )

                    for cid, conf, box in zip(
                        class_ids,
                        confidences,
                        boxes
                    ):
                        name = (
                            model_names[int(cid)]
                            if isinstance(
                                model_names,
                                dict
                            )
                            else model_names[
                                int(cid)
                            ]
                        )

                        rows.append({
                            "class": metric_name(
                                name
                            ),
                            "confidence": float(
                                conf
                            ),
                            "box": box,
                        })

                predictions[
                    key
                ] = rows

            progress.update(
                len(chunk)
            )

            del results

    finally:
        progress.close()

        if getattr(
            detector,
            "model",
            None
        ) is not None:
            detector.model.to(
                "cpu"
            )

        del detector
        gc.collect()
        torch.cuda.empty_cache()

    return predictions


def pairwise_iou(
    pred_box,
    gt_boxes
):
    if len(gt_boxes) == 0:
        return np.empty(
            0,
            dtype=float
        )

    tl = np.maximum(
        pred_box[:2],
        gt_boxes[:, :2]
    )

    br = np.minimum(
        pred_box[2:],
        gt_boxes[:, 2:]
    )

    wh = np.maximum(
        0.0,
        br - tl
    )

    inter = (
        wh[:, 0]
        * wh[:, 1]
    )

    pred_area = (
        max(
            0.0,
            pred_box[2]
            - pred_box[0]
        )
        * max(
            0.0,
            pred_box[3]
            - pred_box[1]
        )
    )

    gt_area = (
        np.maximum(
            0.0,
            gt_boxes[:, 2]
            - gt_boxes[:, 0]
        )
        * np.maximum(
            0.0,
            gt_boxes[:, 3]
            - gt_boxes[:, 1]
        )
    )

    union = (
        pred_area
        + gt_area
        - inter
    )

    return np.divide(
        inter,
        union,
        out=np.zeros_like(
            inter
        ),
        where=union > 0
    )


def match_class(
    gt_boxes,
    prediction_rows,
    threshold
):
    rows = [
        x
        for x in prediction_rows
        if x[
            "confidence"
        ] >= threshold
    ]

    rows = sorted(
        rows,
        key=lambda x:
            -x["confidence"]
    )

    matched = np.zeros(
        len(gt_boxes),
        dtype=bool
    )

    tp = 0
    fp = 0

    for pred in rows:
        ious = pairwise_iou(
            np.asarray(
                pred["box"],
                dtype=float
            ),
            gt_boxes
        )

        if len(ious):
            ious[
                matched
            ] = -1

            best = int(
                np.argmax(
                    ious
                )
            )

            if ious[
                best
            ] >= MATCH_IOU:
                matched[
                    best
                ] = True

                tp += 1
                continue

        fp += 1

    fn = int(
        (~matched).sum()
    )

    return (
        tp,
        fp,
        fn
    )


def score_metric_rows(
    metric_df
):
    rows = []

    for row in metric_df.to_dict(
        "records"
    ):
        class_name = row[
            "class"
        ]

        ratings = GALVAT_RATINGS[
            class_name
        ]

        precision = float(
            row["precision"]
        )

        recall = float(
            row["recall"]
        )

        fn_remaining, fn_possible, fn_score = remaining_area(
            1 - recall,
            ratings["fn"]
        )

        fp_remaining, fp_possible, fp_score = remaining_area(
            1 - precision,
            ratings["fp"]
        )

        row[
            "fn_score"
        ] = fn_score

        row[
            "fp_score"
        ] = fp_score

        row[
            "earned_weighted_points"
        ] = (
            fn_remaining
            + fp_remaining
        )

        row[
            "possible_weight"
        ] = (
            fn_possible
            + fp_possible
        )

        row[
            "class_score"
        ] = (
            row[
                "earned_weighted_points"
            ]
            / row[
                "possible_weight"
            ]
        )

        rows.append(
            row
        )

    result = pd.DataFrame(
        rows
    )

    overall = (
        result[
            "earned_weighted_points"
        ].sum()
        / result[
            "possible_weight"
        ].sum()
    )

    return float(
        overall
    ), result


def evaluate_threshold_grid(
    gt_cache,
    prediction_cache,
    thresholds
):
    outputs = []

    for threshold in thresholds:
        metric_rows = []

        for class_name in RATED_CLASSES:
            tp = fp = fn = 0

            for image_name, gt_by_class in gt_cache.items():
                gt_boxes = gt_by_class.get(
                    class_name,
                    np.empty(
                        (0, 4),
                        dtype=float
                    )
                )

                image_preds = [
                    x
                    for x in prediction_cache.get(
                        image_name,
                        []
                    )
                    if x[
                        "class"
                    ] == class_name
                ]

                a, b, c = match_class(
                    gt_boxes,
                    image_preds,
                    threshold
                )

                tp += a
                fp += b
                fn += c

            precision = (
                tp / (tp + fp)
                if tp + fp
                else 0.0
            )

            recall = (
                tp / (tp + fn)
                if tp + fn
                else 0.0
            )

            metric_rows.append({
                "class": class_name,
                "threshold": float(
                    threshold
                ),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": precision,
                "recall": recall,
            })

        overall, scored = score_metric_rows(
            pd.DataFrame(
                metric_rows
            )
        )

        scored[
            "global_galvat"
        ] = overall

        outputs.append(
            scored
        )

    return pd.concat(
        outputs,
        ignore_index=True
    )


def select_class_thresholds(
    threshold_df
):
    selected = []

    for class_name, group in threshold_df.groupby(
        "class"
    ):
        best = group.sort_values(
            [
                "class_score",
                "precision",
                "recall",
                "threshold"
            ],
            ascending=[
                False,
                False,
                False,
                False
            ]
        ).iloc[0]

        selected.append(
            best.to_dict()
        )

    selected = pd.DataFrame(
        selected
    )

    overall = (
        selected[
            "earned_weighted_points"
        ].sum()
        / selected[
            "possible_weight"
        ].sum()
    )

    thresholds = {
        row["class"]:
            float(
                row["threshold"]
            )
        for row in selected.to_dict(
            "records"
        )
    }

    return (
        float(overall),
        selected,
        thresholds
    )


def evaluate_threshold_map(
    gt_cache,
    prediction_cache,
    threshold_map
):
    metric_rows = []

    for class_name in RATED_CLASSES:
        tp = fp = fn = 0

        threshold = float(
            threshold_map[
                class_name
            ]
        )

        for image_name, gt_by_class in gt_cache.items():
            gt_boxes = gt_by_class.get(
                class_name,
                np.empty(
                    (0, 4),
                    dtype=float
                )
            )

            image_preds = [
                x
                for x in prediction_cache.get(
                    image_name,
                    []
                )
                if x[
                    "class"
                ] == class_name
            ]

            a, b, c = match_class(
                gt_boxes,
                image_preds,
                threshold
            )

            tp += a
            fp += b
            fn += c

        precision = (
            tp / (tp + fp)
            if tp + fp
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn
            else 0.0
        )

        metric_rows.append({
            "class": class_name,
            "threshold": threshold,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
        })

    return score_metric_rows(
        pd.DataFrame(
            metric_rows
        )
    )


print("Evaluation helpers ready.")

In [ ]:
# ============================================================
# 11. VALIDATION-ONLY CHECKPOINT / NMS / THRESHOLD SELECTION
# ============================================================

def epoch_number(path):
    digits = "".join(
        ch
        for ch in path.stem
        if ch.isdigit()
    )

    return (
        int(digits)
        if digits
        else 10**9
    )


periodic = sorted(
    (
        TRAIN_DIR
        / "weights"
    ).glob("epoch*.pt"),
    key=epoch_number
)

candidates = []
seen_hashes = set()

for path in [
    *periodic,
    BEST_PT,
    LAST_PT
]:
    if not path.exists():
        continue

    h = file_sha1(
        path
    )

    if h in seen_hashes:
        continue

    seen_hashes.add(h)

    candidates.append(
        path
    )


print(
    "Checkpoint candidates:",
    [
        p.name
        for p in candidates
    ]
)

PAIR_CACHE = (
    SELECTION_DIR
    / "pair_cache"
)

PAIR_CACHE.mkdir(
    parents=True,
    exist_ok=True
)

leaderboard_rows = []

for checkpoint in candidates:
    ck_sha1 = file_sha1(
        checkpoint
    )

    for nms_iou in NMS_IOU_VALUES:
        pair_name = (
            f"{checkpoint.stem}_"
            f"{ck_sha1[:10]}_"
            f"nms_{str(nms_iou).replace('.', 'p')}"
        )

        pair_dir = (
            PAIR_CACHE
            / pair_name
        )

        pair_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        threshold_csv = (
            pair_dir
            / "threshold_grid.csv"
        )

        if threshold_csv.exists():
            threshold_df = pd.read_csv(
                threshold_csv
            )
        else:
            preds = collect_predictions(
                checkpoint,
                "valid",
                nms_iou
            )

            threshold_df = evaluate_threshold_grid(
                GT_CACHE["valid"],
                preds,
                CONF_VALUES
            )

            threshold_df.to_csv(
                threshold_csv,
                index=False
            )

            del preds
            gc.collect()
            torch.cuda.empty_cache()

        (
            pair_galvat,
            selected_rows,
            thresholds
        ) = select_class_thresholds(
            threshold_df
        )

        selected_rows.to_csv(
            pair_dir
            / "selected_per_class.csv",
            index=False
        )

        (
            pair_dir
            / "selected_thresholds.json"
        ).write_text(
            json.dumps(
                thresholds,
                indent=2
            )
        )

        leaderboard_rows.append({
            "checkpoint": str(
                checkpoint
            ),
            "checkpoint_name":
                checkpoint.name,
            "checkpoint_sha1":
                ck_sha1,
            "nms_iou":
                nms_iou,
            "validation_shared44_galvat":
                pair_galvat,
        })


leaderboard_df = pd.DataFrame(
    leaderboard_rows
).sort_values(
    "validation_shared44_galvat",
    ascending=False
)

leaderboard_df.to_csv(
    SELECTION_DIR
    / "validation_leaderboard.csv",
    index=False
)

display(
    leaderboard_df.head(20)
)


winner = leaderboard_df.iloc[0]

SELECTED_PT = Path(
    winner["checkpoint"]
)

SELECTED_PT_SHA1 = str(
    winner["checkpoint_sha1"]
)

SELECTED_NMS_IOU = float(
    winner["nms_iou"]
)

selected_pair_name = (
    f"{SELECTED_PT.stem}_"
    f"{SELECTED_PT_SHA1[:10]}_"
    f"nms_{str(SELECTED_NMS_IOU).replace('.', 'p')}"
)

selected_pair_dir = (
    PAIR_CACHE
    / selected_pair_name
)

CLASS_THRESHOLDS = json.loads(
    (
        selected_pair_dir
        / "selected_thresholds.json"
    ).read_text()
)

SELECTED_VAL_GALVAT = float(
    winner[
        "validation_shared44_galvat"
    ]
)

(
    SELECTION_DIR
    / "selected_policy.json"
).write_text(
    json.dumps({
        "checkpoint":
            str(SELECTED_PT),
        "checkpoint_sha1":
            SELECTED_PT_SHA1,
        "nms_iou":
            SELECTED_NMS_IOU,
        "class_thresholds":
            CLASS_THRESHOLDS,
        "validation_shared44_galvat":
            SELECTED_VAL_GALVAT,
        "test_used_for_selection":
            False,
    }, indent=2)
)

print("Selected:", SELECTED_PT)
print("Validation GALVAT:", SELECTED_VAL_GALVAT)
print("NMS IoU:", SELECTED_NMS_IOU)

## Fair Trial 8 comparison

Trial 8 is not compared against its old `0.730` test score as though the datasets were identical.

Instead, if the Trial-8 selected checkpoint is available, the notebook recalibrates **Trial 8 on the new validation set only**, then evaluates it once on the exact same new frozen test images. Because Trial 8 lacks the new measuring-spoon class, the primary comparison is the shared 44-class GALVAT score.

In [ ]:
# ============================================================
# 12. CALIBRATE TRIAL 8 ON NEW VALIDATION SET (OPTIONAL)
# ============================================================

trial8_available = TRIAL8_PT.exists()

if not trial8_available:
    print(
        "Trial-8 checkpoint not found; skipping fair baseline."
    )

else:
    trial8_rows = []

    for nms_iou in NMS_IOU_VALUES:
        cache_csv = (
            SELECTION_DIR
            / f"trial8_newval_nms_{str(nms_iou).replace('.', 'p')}.csv"
        )

        if cache_csv.exists():
            grid = pd.read_csv(
                cache_csv
            )
        else:
            preds8 = collect_predictions(
                TRIAL8_PT,
                "valid",
                nms_iou
            )

            grid = evaluate_threshold_grid(
                GT_CACHE["valid"],
                preds8,
                CONF_VALUES
            )

            grid.to_csv(
                cache_csv,
                index=False
            )

            del preds8
            gc.collect()
            torch.cuda.empty_cache()

        score8, rows8, thresholds8 = select_class_thresholds(
            grid
        )

        trial8_rows.append({
            "nms_iou": nms_iou,
            "validation_shared44_galvat":
                score8,
            "thresholds":
                thresholds8,
        })


    trial8_best = sorted(
        trial8_rows,
        key=lambda x:
            x[
                "validation_shared44_galvat"
            ],
        reverse=True
    )[0]

    TRIAL8_NMS_IOU = float(
        trial8_best[
            "nms_iou"
        ]
    )

    TRIAL8_THRESHOLDS = (
        trial8_best[
            "thresholds"
        ]
    )

    TRIAL8_VAL_GALVAT = float(
        trial8_best[
            "validation_shared44_galvat"
        ]
    )

    (
        SELECTION_DIR
        / "trial8_new_validation_policy.json"
    ).write_text(
        json.dumps({
            "checkpoint":
                str(TRIAL8_PT),
            "nms_iou":
                TRIAL8_NMS_IOU,
            "class_thresholds":
                TRIAL8_THRESHOLDS,
            "validation_shared44_galvat":
                TRIAL8_VAL_GALVAT,
            "test_used_for_selection":
                False,
        }, indent=2)
    )

    print(
        "Trial-8 recalibrated validation GALVAT:",
        TRIAL8_VAL_GALVAT
    )

    print(
        "Trial-8 selected NMS:",
        TRIAL8_NMS_IOU
    )

In [ ]:
# ============================================================
# 13. ONE-SHOT FROZEN TEST — TRIAL 9 + FAIR TRIAL 8 BASELINE
# ============================================================

TEST_LOCK = TEST_DIR / "FROZEN_TEST_COMPLETE.json"

if TEST_LOCK.exists():
    summary = json.loads(
        (
            TEST_DIR
            / "frozen_test_summary.json"
        ).read_text()
    )

    print(
        "Frozen test already evaluated."
    )

else:
    trial9_test_preds = collect_predictions(
        SELECTED_PT,
        "test",
        SELECTED_NMS_IOU
    )

    (
        TRIAL9_TEST_GALVAT,
        trial9_test_rows
    ) = evaluate_threshold_map(
        GT_CACHE["test"],
        trial9_test_preds,
        CLASS_THRESHOLDS
    )

    trial9_test_rows.to_csv(
        TEST_DIR
        / "trial9_shared44_per_class.csv",
        index=False
    )

    summary = {
        "split_manifest_sha1":
            SPLIT_MANIFEST_SHA1,
        "trial9_checkpoint":
            str(SELECTED_PT),
        "trial9_checkpoint_sha1":
            SELECTED_PT_SHA1,
        "trial9_nms_iou":
            SELECTED_NMS_IOU,
        "trial9_shared44_galvat":
            TRIAL9_TEST_GALVAT,
        "test_tuning_performed":
            False,
    }


    if trial8_available:
        trial8_test_preds = collect_predictions(
            TRIAL8_PT,
            "test",
            TRIAL8_NMS_IOU
        )

        (
            TRIAL8_TEST_GALVAT,
            trial8_test_rows
        ) = evaluate_threshold_map(
            GT_CACHE["test"],
            trial8_test_preds,
            TRIAL8_THRESHOLDS
        )

        trial8_test_rows.to_csv(
            TEST_DIR
            / "trial8_shared44_per_class.csv",
            index=False
        )

        summary.update({
            "trial8_checkpoint":
                str(TRIAL8_PT),
            "trial8_nms_iou":
                TRIAL8_NMS_IOU,
            "trial8_shared44_galvat":
                TRIAL8_TEST_GALVAT,
            "trial9_minus_trial8_galvat":
                TRIAL9_TEST_GALVAT
                - TRIAL8_TEST_GALVAT,
        })


    (
        TEST_DIR
        / "frozen_test_summary.json"
    ).write_text(
        json.dumps(
            summary,
            indent=2
        )
    )

    TEST_LOCK.write_text(
        json.dumps({
            "completed_at":
                str(
                    datetime.datetime.now()
                ),
            "split_manifest_sha1":
                SPLIT_MANIFEST_SHA1,
            "trial9_checkpoint_sha1":
                SELECTED_PT_SHA1,
            "test_tuning_performed":
                False,
        }, indent=2)
    )


print(
    json.dumps(
        summary,
        indent=2
    )
)

In [ ]:
# ============================================================
# 14. OFFICIAL TRIAL-9 AP / FULL ONTOLOGY METRICS
# ============================================================

selected_model = YOLO(
    str(SELECTED_PT)
)

official_test = selected_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    conf=0.001,
    iou=SELECTED_NMS_IOU,
    device=0,
    plots=True,
    save_json=True,
    project=str(TEST_DIR),
    name="official_trial9_test",
    exist_ok=True,
    verbose=False,
)

official_summary = pd.DataFrame(
    official_test.summary()
)

official_summary.to_csv(
    TEST_DIR
    / "trial9_official_full_ontology_metrics.csv",
    index=False
)

display(
    official_summary
)

# Highlight any classes without GALVAT ratings, e.g. measuring spoon.
if unrated_dataset_classes:
    normalized_class_col = None

    for candidate in [
        "Class",
        "class"
    ]:
        if candidate in official_summary.columns:
            normalized_class_col = candidate
            break

    if normalized_class_col:
        extra_rows = official_summary[
            official_summary[
                normalized_class_col
            ].astype(str).map(
                metric_name
            ).isin(
                unrated_dataset_classes
            )
        ]

        print("Unrated Trial-9 classes:")
        display(extra_rows)

In [ ]:
# ============================================================
# 15. ASSISTIVE OUTPUT BURDEN + FRESH BACKGROUND ROI
# ============================================================

def accepted_count_rows(
    prediction_cache,
    thresholds
):
    rows = []

    for image_name, preds in prediction_cache.items():
        accepted = []

        for p in preds:
            class_name = p[
                "class"
            ]

            # New/unrated classes use a conservative default until
            # formal class-specific calibration is defined.
            threshold = thresholds.get(
                class_name,
                0.50
            )

            if p[
                "confidence"
            ] >= threshold:
                accepted.append(
                    class_name
                )

        rows.append({
            "image": image_name,
            "accepted_detections":
                len(accepted),
            "classes":
                ",".join(
                    accepted
                ),
        })

    return pd.DataFrame(
        rows
    )


if "trial9_test_preds" not in globals():
    trial9_test_preds = collect_predictions(
        SELECTED_PT,
        "test",
        SELECTED_NMS_IOU
    )


spoken_df = accepted_count_rows(
    trial9_test_preds,
    CLASS_THRESHOLDS
)

spoken_df.to_csv(
    TEST_DIR
    / "trial9_spoken_detection_counts.csv",
    index=False
)

SPOKEN_BUDGET = 3

spoken_summary = {
    "images":
        len(spoken_df),
    "within_3_item_budget_rate":
        float(
            (
                spoken_df[
                    "accepted_detections"
                ] <= SPOKEN_BUDGET
            ).mean()
        ),
    "over_3_item_budget_rate":
        float(
            (
                spoken_df[
                    "accepted_detections"
                ] > SPOKEN_BUDGET
            ).mean()
        ),
}

(
    TEST_DIR
    / "spoken_budget_summary.json"
).write_text(
    json.dumps(
        spoken_summary,
        indent=2
    )
)

print(
    json.dumps(
        spoken_summary,
        indent=2
    )
)

# Packaging policy

We do **not** fuse YOLO26s and SAM 3 into a single network.

The deployment bundle has two layers:

**Detector-only profile:** selected YOLO26s checkpoint + ONNX + ontology + NMS + class thresholds. This is the portable semantic detector.

**Detector + SAM 3 reference profile:** the same YOLO detector plus a tracking policy that seeds SAM 3 from YOLO boxes, limits active tracks, periodically refreshes with YOLO, and never allows SAM 3 to redefine the class identity.

For assistive use, the tracker should normally follow only the few objects that matter to the user's current task rather than every detected object in the scene.

In [ ]:
# ============================================================
# 16. EXPORT YOLO26s + BUILD DEPLOYMENT BUNDLE
# ============================================================

DETECTOR_DIR = BUNDLE_DIR / "detector"
TRACKER_DIR = BUNDLE_DIR / "tracker"
RUNTIME_DIR = BUNDLE_DIR / "runtime"

for p in [
    DETECTOR_DIR,
    TRACKER_DIR,
    RUNTIME_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


SELECTED_COPY = (
    DETECTOR_DIR
    / "galvat_trial9_yolo26s_selected.pt"
)

shutil.copy2(
    SELECTED_PT,
    SELECTED_COPY
)


export_model = YOLO(
    str(SELECTED_COPY)
)

exported = Path(
    export_model.export(
        format="onnx",
        imgsz=IMGSZ,
        opset=12,
        simplify=True,
        dynamic=False,
        nms=False,
        batch=1,
        device=0,
    )
)

ONNX_PATH = (
    DETECTOR_DIR
    / "galvat_trial9_yolo26s.onnx"
)

if exported.resolve() != ONNX_PATH.resolve():
    shutil.copy2(
        exported,
        ONNX_PATH
    )

assert ONNX_PATH.exists()


class_map = {
    i: name
    for i, name in enumerate(
        NAMES
    )
}

(
    DETECTOR_DIR
    / "classes.json"
).write_text(
    json.dumps(
        class_map,
        indent=2
    )
)

(
    DETECTOR_DIR
    / "class_thresholds.json"
).write_text(
    json.dumps(
        CLASS_THRESHOLDS,
        indent=2
    )
)

detector_config = {
    "architecture":
        "YOLO26s",
    "input_size":
        [640, 640],
    "input_layout":
        "NCHW",
    "semantic_authority":
        True,
    "onnx_has_embedded_nms":
        False,
    "nms_iou":
        SELECTED_NMS_IOU,
    "max_detections":
        MAX_DET,
    "confidence_policy":
        "per_class_validation_selected_for_44_rated_classes",
    "unrated_class_default_confidence":
        0.50,
    "primary_metric":
        "shared_44_class_GALVAT",
}

(
    DETECTOR_DIR
    / "detector_config.json"
).write_text(
    json.dumps(
        detector_config,
        indent=2
    )
)


tracker_config = {
    "tracker":
        "SAM3_video",
    "role":
        "temporal_localization_only",
    "semantic_source":
        "YOLO26s",
    "seed_policy":
        "YOLO_box_prompt",
    "allow_tracker_to_change_class":
        False,
    "max_active_tracks":
        3,
    "detector_refresh_every_frames":
        5,
    "early_refresh_conditions": [
        "track_missing",
        "track_geometry_implausible",
        "user_changes_target",
        "detector_tracker_disagreement"
    ],
    "association_policy":
        "same_YOLO_class_then_IoU",
    "reassociation_iou":
        0.30,
    "terminate_after_failed_refreshes":
        2,
    "output_policy":
        "announce_state_changes_or_requested_objects_not_every_frame",
    "sam3_checkpoint_distribution":
        "external_or_gated_not_bundled_by_this_notebook",
    "reference_runtime":
        "CUDA_PyTorch",
}

(
    TRACKER_DIR
    / "sam3_tracker_config.json"
).write_text(
    json.dumps(
        tracker_config,
        indent=2
    )
)


pipeline_config = {
    "pipeline_name":
        "GALVAT_Trial9_Detect_Track",
    "detector":
        "detector/galvat_trial9_yolo26s.onnx",
    "detector_thresholds":
        "detector/class_thresholds.json",
    "tracker_config":
        "tracker/sam3_tracker_config.json",
    "active_object_budget":
        3,
    "selection_order": [
        "explicit_user_target",
        "assistive_priority",
        "detector_confidence"
    ],
    "profiles": {
        "detector_only_magic_leap_candidate": {
            "enabled":
                True,
            "description":
                "YOLO26s ONNX plus thresholds; SAM3 not required."
        },
        "detector_plus_sam3_cuda_reference": {
            "enabled":
                True,
            "description":
                "Research reference pipeline for CUDA-capable companion compute."
        }
    }
}

(
    RUNTIME_DIR
    / "pipeline_config.json"
).write_text(
    json.dumps(
        pipeline_config,
        indent=2
    )
)


README = """
GALVAT Trial 9 Deployment Bundle
================================

1. YOLO26s is the semantic detector.
2. Apply class-aware NMS using detector_config.json.
3. Apply the per-class thresholds in class_thresholds.json.
4. Rank detections by user intent / assistive importance and keep a small active set.
5. Seed SAM 3 tracking from the selected YOLO boxes.
6. SAM 3 only propagates localization/masks. It must not change the YOLO class label.
7. Refresh YOLO periodically and whenever tracking becomes uncertain.
8. Reassociate only when the class agrees and geometry is plausible.
9. Limit active tracks to reduce compute and spoken-output burden.

Magic Leap note:
The detector-only ONNX profile is the portable deployment artifact.
The official SAM 3 video predictor is a CUDA/PyTorch reference implementation,
so SAM 3 should initially run on CUDA-capable companion/host compute rather
than being assumed to run natively on Magic Leap 2.
"""

(
    BUNDLE_DIR
    / "README_DEPLOYMENT.txt"
).write_text(
    README.strip()
)


def sha1(path):
    h = hashlib.sha1()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


bundle_manifest = {
    "trial":
        9,
    "split_manifest_sha1":
        SPLIT_MANIFEST_SHA1,
    "selected_checkpoint_sha1":
        sha1(SELECTED_COPY),
    "onnx_sha1":
        sha1(ONNX_PATH),
    "onnx_size_mb":
        ONNX_PATH.stat().st_size / 1e6,
    "selected_validation_shared44_galvat":
        SELECTED_VAL_GALVAT,
    "frozen_test_summary":
        summary,
    "created_at":
        str(
            datetime.datetime.now()
        ),
}

(
    BUNDLE_DIR
    / "bundle_manifest.json"
).write_text(
    json.dumps(
        bundle_manifest,
        indent=2
    )
)


BUNDLE_ZIP = (
    RUN_ROOT
    / "GALVAT_Trial9_Deployment_Bundle.zip"
)

if BUNDLE_ZIP.exists():
    BUNDLE_ZIP.unlink()

with zipfile.ZipFile(
    BUNDLE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in BUNDLE_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    path.relative_to(
                        BUNDLE_DIR
                    )
                )
            )


print("ONNX:", ONNX_PATH)
print("Deployment bundle:", BUNDLE_ZIP)
print(
    json.dumps(
        bundle_manifest,
        indent=2
    )
)

In [ ]:
# ============================================================
# RELOAD TRIAL 9 MODEL + EXPORT WITH MAX 40 DETECTIONS
# ============================================================

!pip -q install ultralytics==8.4.102 onnx onnxruntime

from pathlib import Path
from google.colab import drive
from ultralytics import YOLO
import shutil
import onnxruntime as ort

drive.mount("/content/drive")

# ------------------------------------------------------------
# Find the Trial 9 selected/best checkpoint in Drive
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/content/drive/MyDrive/galvat_trial9_training_runs/"
    "galvat_trial9_yolo26s_rf_v2"
)

# First look for the final selected model.
candidates = list(RUN_ROOT.rglob("*selected*.pt"))

# If that name doesn't exist, fall back to best.pt.
if not candidates:
    candidates = list(RUN_ROOT.rglob("best.pt"))

# Last fallback: any .pt model in the run.
if not candidates:
    candidates = list(RUN_ROOT.rglob("*.pt"))

assert candidates, f"No .pt checkpoint found under {RUN_ROOT}"

# Show what was found.
print("Found checkpoints:")
for p in candidates:
    print(" -", p)

MODEL_PT = candidates[0]

print()
print("Using:", MODEL_PT)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = YOLO(str(MODEL_PT))

# ------------------------------------------------------------
# Export ONNX with only 40 detections
# ------------------------------------------------------------

onnx_path = model.export(
    format="onnx",
    imgsz=640,
    batch=1,
    dynamic=False,
    simplify=True,
    nms=False,
    max_det=40
)

onnx_path = Path(onnx_path)

print()
print("Temporary ONNX:", onnx_path)

# ------------------------------------------------------------
# Save a permanent copy to Google Drive
# ------------------------------------------------------------

EXPORT_DIR = RUN_ROOT / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_ONNX = (
    EXPORT_DIR
    / "galvat_trial9_yolo26s_maxdet40.onnx"
)

shutil.copy2(
    onnx_path,
    FINAL_ONNX
)

print("Saved permanently to:")
print(FINAL_ONNX)

# ------------------------------------------------------------
# Verify ONNX input/output shapes
# ------------------------------------------------------------

session = ort.InferenceSession(
    str(FINAL_ONNX),
    providers=["CPUExecutionProvider"]
)

print()
print("INPUT:")
for x in session.get_inputs():
    print(x.name, x.shape)

print()
print("OUTPUT:")
for x in session.get_outputs():
    print(x.name, x.shape)

## SAM 3 tracking implementation plan after training

The next integration step should be a **separate video/runtime notebook**, not part of the training loop.

That notebook should load the Trial-9 detector bundle, run YOLO on a refresh cadence, seed SAM 3 using YOLO detections, propagate tracks between refresh frames, measure drift/reacquisition, and report end-to-end latency.

The key design rule is that SAM 3 improves **temporal continuity**, not semantic recognition. If YOLO says `medicine bottle`, the track remains `medicine bottle` until a later YOLO refresh confirms or terminates it.

This gives us a clean experimental decomposition:

`Trial-9 static detector quality` + `SAM-3 temporal tracking quality` + `full assistive-system latency/output burden`

In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

p = Path(
    "/content/drive/MyDrive/galvat_trial9_training_runs/"
    "galvat_trial9_yolo26s_rf_v2/"
    "deployment_bundle/detector/class_thresholds.json"
)

thresholds = json.loads(p.read_text())

print(json.dumps(thresholds, indent=2))

In [ ]:
!pip uninstall -y ultralytics
!pip install -U ultralytics roboflow

import ultralytics
print("Ultralytics version:", ultralytics.__version__)

In [ ]:
# ============================================================
# Upload GALVAT Trial 9 model to Roboflow for Label Assist
# ============================================================
import os
from pathlib import Path
from google.colab import drive
from roboflow import Roboflow

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Roboflow project info
# ------------------------------------------------------------

WORKSPACE = "aud-research"
PROJECT = "objects-ow83t-ylj7g"

# IMPORTANT:
# Attach the model to the Roboflow dataset version corresponding
# to the class ontology used to train Trial 9.
DATASET_VERSION = 1

# ------------------------------------------------------------
# 3. Locate Trial 9 PyTorch weights
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/content/drive/MyDrive/galvat_trial9_training_runs/"
    "galvat_trial9_yolo26s_rf_v2"
)

candidates = list(RUN_ROOT.rglob("epoch110.pt"))

if not candidates:
    candidates = list(RUN_ROOT.rglob("best.pt"))

assert candidates, f"No Trial 9 .pt weights found under {RUN_ROOT}"

MODEL_PT = candidates[0]

print("Using model:")
print(MODEL_PT)

# ------------------------------------------------------------
# 4. Roboflow deploy() expects a training/run directory
#    containing weights/best.pt, so make a clean temporary folder
# ------------------------------------------------------------

UPLOAD_DIR = Path("/content/galvat_trial9_roboflow")
WEIGHTS_DIR = UPLOAD_DIR / "weights"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Roboflow generally looks for weights/best.pt
import shutil

RF_BEST = WEIGHTS_DIR / "best.pt"
shutil.copy2(MODEL_PT, RF_BEST)

print("\nPrepared Roboflow upload folder:")
print(UPLOAD_DIR)
print("Weight file:")
print(RF_BEST)

# ------------------------------------------------------------
# 5. Connect to Roboflow
# ------------------------------------------------------------

# Recommended: store ROBOFLOW_API_KEY in Colab Secrets or env
API_KEY = os.environ.get("ROBOFLOW_API_KEY")

if API_KEY is None:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")

assert API_KEY, "ROBOFLOW_API_KEY not found."

rf = Roboflow(api_key=API_KEY)

project = (
    rf.workspace(WORKSPACE)
      .project(PROJECT)
)

version = project.version(DATASET_VERSION)

# ------------------------------------------------------------
# 6. Upload Trial 9 weights
# ------------------------------------------------------------

print("\nStarting upload...")

version.deploy(
    model_type="yolo26",
    model_path=str(UPLOAD_DIR)
)

print("\nUpload request completed.")
print("Check the Roboflow version's Deploy / Model section.")
print("Once processing finishes, it should become available for Label Assist.")